In [66]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# The following is vibe-coded with Gemini, FIXED VERSION BY KEVIN LIM

torch.set_default_dtype(torch.float64) # MAY WANT TO CHANGE LATER

def calculate_J(R1, R2, S0, theta):
    """
    NOTE: Remember to input sigs2 correctly
    """
    
    # 1. Slicing the parameters
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1_raw = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2_raw = theta[idx : idx + R2]; idx += R2

    nus1 = torch.cat([nus1, torch.tensor([S0])])

    nus2 = torch.cat([torch.tensor([S0]), nus2])

    # Enforce strict positivity on sigmas using softplus
    sigs1 = torch.nn.functional.softplus(sigs1_raw) + 1e-6
    sigs2 = torch.nn.functional.softplus(sigs2_raw) + 1e-6

    # =========================================================================
    # STAGE 1: Calculate Unit Base Coefficients (Setting lam1 = 1, lam2 = 1)
    # =========================================================================

    # Left Wing Unit Base
    c_v1_unit = []
    dist = nus1[1] - nus1[0]
    s0 = sigs1[0]
    c_v1_unit.append((-torch.exp(-dist/s0), torch.exp(dist/s0)))

    for j in range(R1 - 1):
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1.0/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1[j+2] - nus1[j+1]
        c_v1_unit.append((0.5 * (P - sigs1[j+1]*S) * torch.exp(-dist/sigs1[j+1]),
                          0.5 * (P + sigs1[j+1]*S) * torch.exp(dist/sigs1[j+1])))

    # Right Wing Unit Base
    c_v2_unit = [None] * R2
    dist = nus2[-1] - nus2[-2]
    c_v2_unit[-1] = ((torch.exp(dist/sigs2[-1]), -torch.exp(-dist/sigs2[-1])))

    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1.0/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2[j] - nus2[j-1]
        c_v2_unit[j-1] = (0.5 * (P - sigs2[j-1]*S) * torch.exp(dist/sigs2[j-1]),
                          0.5 * (P + sigs2[j-1]*S) * torch.exp(-dist/sigs2[j-1]))

    # Verification of this method to compute c values

    # c_v1_test = []
    # c_v1_test.append((-torch.exp(nus1[0]/sigs1[0]), torch.exp(-nus1[0]/sigs1[0])))
    # for j in range(R1 - 1):
    #     c_v1_test.append((  0.5 * ((1 + sigs1[j+1]/sigs1[j])*c_v1_test[j][0]*torch.exp(-nus1[j+1]/sigs1[j]) + (1 - sigs1[j+1]/sigs1[j])*c_v1_test[j][1]*torch.exp(nus1[j+1]/sigs1[j])) / torch.exp(-nus1[j+1]/sigs1[j+1]),
    #                         0.5 * ((1 - sigs1[j+1]/sigs1[j])*c_v1_test[j][0]*torch.exp(-nus1[j+1]/sigs1[j]) + (1 + sigs1[j+1]/sigs1[j])*c_v1_test[j][1]*torch.exp(nus1[j+1]/sigs1[j])) / torch.exp(nus1[j+1]/sigs1[j+1])))
    # for i in range(R1):
    #     print("old:", (c_v1_test[i][0] * torch.exp(-nus1[i+1]/sigs1[i]),  c_v1_test[i][1] * torch.exp(nus1[i+1]/sigs1[i])))
    #     print("new:", (c_v1_unit[i][0], c_v1_unit[i][1]))

    # c_v2_test = [None] * R2
    # c_v2_test[-1] = (torch.exp(nus2[-1]/sigs2[-1]), -torch.exp(-nus2[-1]/sigs2[-1]))
    # for j in range(R2 - 1, 0, -1):
    #     c_v2_test[j-1] = (  0.5 * ((1 + sigs2[j-1]/sigs2[j])*c_v2_test[j][0]*torch.exp(-nus2[j]/sigs2[j]) + (1 - sigs2[j-1]/sigs2[j])*c_v2_test[j][1]*torch.exp(nus2[j]/sigs2[j])) / torch.exp(-nus2[j]/sigs2[j-1]),
    #                         0.5 * ((1 - sigs2[j-1]/sigs2[j])*c_v2_test[j][0]*torch.exp(-nus2[j]/sigs2[j]) + (1 + sigs2[j-1]/sigs2[j])*c_v2_test[j][1]*torch.exp(nus2[j]/sigs2[j])) / torch.exp(nus2[j]/sigs2[j-1]))
    # for i in range(R2):
    #     print("old:", (c_v2_test[i][0] * torch.exp(-nus2[i]/sigs2[i]),  c_v2_test[i][1] * torch.exp(nus2[i]/sigs2[i])))
    #     print("new:", (c_v2_unit[i][0], c_v2_unit[i][1]))
    

    # =========================================================================
    # STAGE 2: Evaluate Unit Wings at S0 and Solve for Viable Lambdas
    # =========================================================================
    
    # Left wing unit value at S0
    v1 = c_v1_unit[-1][0] + c_v1_unit[-1][1]

    # Right wing unit value at S0
    v2 = c_v2_unit[0][0] + c_v2_unit[0][1]

    # Derivatives at left and right of S0
    Dk_v1 = (1.0/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][1])

    Dk_v2 = (1.0/sigs2[0]) * (-c_v2_unit[0][0] + c_v2_unit[0][1])

    # Define viable lambdas
    lam1 = v2 / (Dk_v1 * v2 - v1 * Dk_v2)

    lam2 = v1 / (Dk_v1 * v2 - v1 * Dk_v2)

    # =========================================================================
    # STAGE 3: Scale Unit Coefficients to Final Viable Coefficients
    # =========================================================================
    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    print(c_v2[0][0] + c_v2[0][1])

    # Plot graph of curve
    # x = torch.linspace(1150, 1350, 1000)
    # y = []
    # for i in x:
    #     if i <= S0:
    #         index = torch.searchsorted(nus1, i, side = 'left') - 1
    #         dist = nus1[index + 1] - i
    #         y.append((c_v1[index][0] * torch.exp(dist/sigs1[index])  + c_v1[index][1] * torch.exp(-dist/sigs1[index])).item() + S0 - i)
    #     else:
    #         index = torch.searchsorted(nus2, i, side = 'left') - 1
    #         dist = i - nus2[index]
    #         y.append((c_v2[index][0] * torch.exp(-dist/sigs2[index])  + c_v2[index][1] * torch.exp(dist/sigs2[index])).item())

    # plt.plot(x,y)

    # =========================================================================
    # STAGE 4: Calculate Smoothness Jumps
    # =========================================================================
    jumps = []

    # Internal Left Wing Jumps
    for j in range(R1 - 1):
        v_sec_left = (1.0/sigs1[j]**2) * (c_v1[j][0] + c_v1[j][1])
        dist = nus1[j+2] - nus1[j+1]
        v_sec_right = (1.0/sigs1[j+1]**2) * (c_v1[j+1][0]*torch.exp(dist/sigs1[j+1]) +
                                            c_v1[j+1][1]*torch.exp(-dist/sigs1[j+1]))
        jumps.append(v_sec_right - v_sec_left)

    # Junction Jump at x_strike
    v_sec_v1_strike = (1.0/sigs1[-1]**2) * (c_v1[-1][0] + c_v1[-1][1])
    v_sec_v2_strike = (1.0/sigs2[0]**2) * (c_v2[0][0] + c_v2[0][1])
    
    jumps.append(v_sec_v2_strike - v_sec_v1_strike)

    # Internal Right Wing Jumps
    for j in range(R2 - 1):
        v_sec_right = (1.0/sigs2[j+1]**2) * (c_v2[j+1][0] + c_v2[j+1][1])
        dist = nus2[j+1] - nus2[j]
        v_sec_left = (1.0/sigs2[j]**2) * (c_v2[j][0]*torch.exp(-dist/sigs2[j+1]) +
                                            c_v2[j][1]*torch.exp(dist/sigs2[j+1]))
        jumps.append(v_sec_right - v_sec_left)

    j_val = torch.sum(torch.stack(jumps) ** 2)
    return j_val, lam1, lam2

# --- Verification execution ---
if __name__ == "__main__":
    R1, R2 = 70, 34 # lengths of sigs1/nus1, and sigs2/nus2

    # Financial Market Conditions
    S0 = 1271.87     # Spot asset price

    # Initialize parameters
    initial_sigs1 = torch.tensor([610.4320996213446, 164.04874695079099, 21.410699478563217, 41.036799095433764, 32.59140099151007, 48.77559067919313, 42.2176372999541, 55.823566769595985, 51.211167181688566, 60.691607330511374, 59.950820637770775, 58.898463641159154, 61.253284806200256, 58.77606559998955, 61.9431978849179, 57.790240203069374, 62.2779153688338, 54.621537161593736, 60.38808834808558, 50.96497274766389, 57.49812984113923, 46.576659253726845, 54.080819207223875, 40.60311710082493, 46.79656147874567, 40.230053767550686, 44.17878499997829, 40.0738197788728, 42.59425882166232, 41.97429249407287, 43.554019179773334, 43.64066412990515, 46.68459597597928, 40.66643248490499, 45.74533322390213, 37.58570160756966, 43.36876861252296, 34.281419881606425, 38.63837902433531, 34.11699307804324, 38.33534468465065, 33.11788846662374, 37.54304297301797, 30.8995436545175, 34.629132576982585, 31.932382438352974, 34.99302768634262, 31.44000481459117, 34.80339761657143, 31.011124152999194, 33.67039062291056, 32.963580616942636, 33.88070107871873, 37.490458108901564, 36.9523875799828, 42.71457754035583, 46.473166992677335, 33.8545330176666, 43.67858033724246, 24.46553767335429, 29.27267842043444, 33.253868886417095, 32.860201483329625, 36.87919843993161, 38.56770262319378, 34.89031672560033, 32.336062185014455, 54.86203086146442, 42.54094598147429, 31.82466458162184])
    initial_sigs2 = torch.tensor([24.02631471182734, 38.31350500726014, 64.22160217291687, 24.945880842361603, 31.475272346451945, 31.167940240782592, 29.04149431455202, 30.22926871363152, 28.245473181139502, 27.3473961133823, 26.078699124964757, 24.718907886876814, 23.35903470115683, 22.454368334929534, 22.002320937211316, 19.569326547161026, 19.315256975829875, 17.756271425921597, 19.392408982133873, 39.61543966355491, 26.387286176796366, 49.89596975113815, 28.527769674700764, 31.874820986884167, 41.72362039958813, 83.04797290752168, 40.34143759307382, 37.0089135084597, 39.353339426345215, 30.273572809409785, 73.52830072545393, 57.33057445441953, 169.4372302588361, 313.0441713343745])
    initial_nus1 = torch.tensor([0, 985.8952130330157, 1103.9, 1105.6299717509714, 1108.9, 1110.9261114114317, 1113.9, 1116.0803462760916, 1118.9, 1121.217532062242, 1123.9, 1126.4524566940233, 1128.9, 1131.482542383951, 1133.9, 1136.5181574323626, 1138.9, 1141.5955696582002, 1143.9, 1146.6439410206626, 1148.9, 1151.6955052869373, 1153.9, 1156.7899943184998, 1158.9, 1161.6254606809712, 1163.9, 1166.5610218554195, 1168.9, 1171.459730260086, 1173.9, 1176.4403363141735, 1178.9, 1181.615546573849, 1183.9, 1186.6888172217348, 1188.9, 1191.7378244235415, 1193.9, 1196.5483693501587, 1198.8, 1201.5325443234078, 1203.8, 1206.5944516093734, 1208.8, 1211.4560716332721, 1213.8, 1216.4902257056644, 1218.8, 1221.5020603603978, 1223.8, 1226.3861829743955, 1228.8, 1231.2336479134826, 1233.8, 1236.1780621637572, 1238.8, 1241.7477258299014, 1243.8, 1247.0561689330698, 1248.8, 1251.2009300066638, 1253.8, 1256.2167752285077, 1258.8, 1261.4846983461696, 1263.8, 1265.7099860650808, 1268.8, 1270.5770900753105])
    initial_nus2 = torch.tensor([1272.6042131350682, 1273.8, 1277.3405408056378, 1278.8, 1281.2353139855318, 1283.8, 1286.1696012675789, 1288.8, 1291.2560021979807, 1293.8, 1296.2794535546955, 1298.8, 1301.213819395613, 1303.7, 1306.2617186435, 1308.7, 1311.232700889095, 1313.7, 1316.810048833819, 1323.7, 1327.0068556961314, 1333.7, 1336.0240360741054, 1338.7, 1343.4698860472697, 1353.7, 1356.277830214803, 1358.7, 1361.5021167401464, 1363.7, 1377.3378359299822, 1388.7, 1503.7243622969859, 2000.0])              
    sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1.0)
    sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1.0)

    theta = torch.cat([
        initial_nus1,
        sigs1_raw,  
        initial_nus2,
        sigs2_raw
    ]).clone().detach().requires_grad_(True)

    # Compute J
    j_score, lam1, lam2 = calculate_J(R1, R2, S0, theta)

    # Run backward pass to confirm gradients track through the linear scaling system
    j_score.backward()

    print("=== Viable Baseline Simulation ===")
    print(f"Smoothness Penalty J: {j_score.item():.8f}")
    print("\nGradients (dJ/dtheta) successfully computed:")
    print(theta.grad)

tensor(16.5130, grad_fn=<AddBackward0>)
=== Viable Baseline Simulation ===
Smoothness Penalty J: 0.00130064

Gradients (dJ/dtheta) successfully computed:
tensor([-3.1428e-11,  2.3544e-09,  3.3687e-07, -2.3518e-07,  5.3887e-08,
        -7.9088e-08,  2.2188e-08, -3.7841e-08,  9.8725e-09, -1.8199e-08,
         1.1576e-09,  1.7716e-09, -3.9493e-09,  4.3234e-09, -5.6044e-09,
         7.8730e-09, -8.7104e-09,  1.7078e-08, -1.3858e-08,  2.6837e-08,
        -2.0537e-08,  4.2609e-08, -3.2693e-08,  8.0419e-08, -4.5916e-08,
         5.4629e-08, -3.6846e-08,  4.2604e-08, -2.8935e-08,  7.2725e-09,
        -1.9263e-08, -1.0921e-09, -3.7351e-08,  9.3571e-08, -8.4870e-08,
         1.8065e-07, -1.4112e-07,  3.0985e-07, -1.7609e-07,  2.1915e-07,
        -2.1271e-07,  3.3510e-07, -2.9800e-07,  6.2989e-07, -3.9377e-07,
         3.3245e-07, -3.8903e-07,  5.7456e-07, -5.6469e-07,  8.2361e-07,
        -6.2595e-07,  1.8321e-07, -2.6288e-07, -9.4899e-07,  1.4983e-07,
        -1.4292e-06, -7.8101e-07,  4.9394e-